In [ ]:
!pip install transformers
!pip install sentence-transformers
!pip install faiss-cpu  # or faiss-gpu if you have a GPU
!pip install langchain  # Optional, if using LangChain
!pip install streamlit
!pip install python-dotenv
!pip install PyPDF2  # If dealing with PDF documents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 24.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.6/396.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.5/290.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/

In [ ]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ''
        for page in reader.pages:
            text += page.extract_text()
    return text

# Usage example:
pdf_path = "hr.pdf"
extracted_text = extract_text_from_pdf(pdf_path)
print(extracted_text[:1000])  # Print the first 1000 characters to check the extraction



FileNotFoundError: [Errno 2] No such file or directory: 'hr.pdf'

In [ ]:
import re

def clean_text(text):
    # Remove multiple spaces, newlines, etc.
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Usage example:
cleaned_text = clean_text(extracted_text)
print(cleaned_text[:1000])  # Check cleaned text


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_text(text, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_text(text)

# Usage example:
chunks = split_text(cleaned_text)
for i, chunk in enumerate(chunks[:3]):  # Preview first 3 chunks
    print(f"Chunk {i+1}:")
    print(chunk[:500])  # Print first 500 characters of each chunk
    print("="*80)



In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained model for sentence embeddings
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def generate_embeddings(chunks):
    return embedding_model.encode(chunks, show_progress_bar=True)

# Usage example:
embeddings = generate_embeddings(chunks)


In [ ]:
import faiss
import numpy as np

def create_faiss_index(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)  # Using a basic L2 (Euclidean distance) index
    index.add(np.array(embeddings).astype('float32'))
    return index

# Usage example:
faiss_index = create_faiss_index(embeddings)


In [ ]:
def embed_query(query, model_name='all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    return model.encode([query], show_progress_bar=False)[0]  # Return the embedding for the query

# Usage example:
user_query = "What are the number of escalation stages of the company?"
query_embedding = embed_query(user_query)


In [ ]:
def retrieve_similar_chunks(query_embedding, index, chunks, top_k=3):
    # Search the FAISS index for the most similar chunks
    distances, indices = index.search(np.array([query_embedding]).astype('float32'), top_k)

    # Fetch the corresponding chunks
    retrieved_chunks = [chunks[i] for i in indices[0]]

    return retrieved_chunks

# Usage example:
retrieved_chunks = retrieve_similar_chunks(query_embedding, faiss_index, chunks, top_k=3)

# Preview the retrieved chunks
for i, chunk in enumerate(retrieved_chunks):
    print(f"Retrieved Chunk {i+1}:")
    print(chunk[:500])  # Show the first 500 characters of each chunk
    print("="*80)


In [ ]:
pip install transformers


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def load_language_model(model_name="google/flan-t5-large"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return tokenizer, model

# Load the model
tokenizer, model = load_language_model()


In [ ]:
# def generate_answer(question, retrieved_chunks, model, tokenizer, max_length=512):
#     # Combine the retrieved chunks as context
#     context = " ".join(retrieved_chunks)

#     # Format the input as a question-answering prompt
#     input_text = f"Context: {context} Question: {question}"

#     # Tokenize the input
#     inputs = tokenizer.encode(input_text, return_tensors="pt", truncation=True)

#     # Generate the answer
#     outputs = model.generate(inputs, max_length=max_length, num_beams=3, early_stopping=True)

#     # Decode and return the generated answer
#     answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     return answer

# # Usage example:
# answer = generate_answer(user_query, retrieved_chunks, model, tokenizer)
# print("Answer:", answer)
def generate_answer(question, retrieved_chunks, model, tokenizer, max_length=1024):
    # Combine the retrieved chunks as context
    context = " ".join(retrieved_chunks)

    # Format the input as a detailed QA prompt with a focus on answering directly from the text
    input_text = f"Use the following information to answer the question clearly: {context} Question: {question}"

    # Tokenize the input
    inputs = tokenizer.encode(input_text, return_tensors="pt", truncation=True)

    # Generate the answer
    outputs = model.generate(inputs, max_length=max_length, num_beams=5, early_stopping=True)

    # Decode and return the generated answer
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# Usage example:
answer = generate_answer(user_query, retrieved_chunks, model, tokenizer)
print("Answer:", answer)


In [ ]:
def summarize_chunks(chunks, model, tokenizer, max_length=512):
    summarized_texts = []
    for chunk in chunks:
        input_text = f"Summarize this text: {chunk}"
        inputs = tokenizer.encode(input_text, return_tensors="pt", truncation=True)
        outputs = model.generate(inputs, max_length=max_length, num_beams=3, early_stopping=True)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summarized_texts.append(summary)
    return " ".join(summarized_texts)

# Summarize first, then use the summary to generate the answer
summarized_chunks = summarize_chunks(retrieved_chunks, model, tokenizer)
answer = generate_answer(user_query, [summarized_chunks], model, tokenizer)
print("Answer from Summarized Chunks:", answer)

In [ ]:
prompt_template = """You are a helpful assistant for the company. Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know.

{context}

Question: {question}
Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)
